# WatermarkingForLLM on Google Colab

This notebook builds **CPRF** (Go) and **PRC** (Rust/maturin), installs Python packages, and runs `app.py` using Colab-native patterns (`%pip`, `notebook_login()`, `pathlib`, `subprocess`).

**Before you start:** **Runtime → Change runtime type → GPU**. Pick a **Python 3.11+** runtime if the menu offers it (required by this project).

In [ ]:
import sys

assert sys.version_info >= (3, 11), "Use Python 3.11+ (Runtime → Change runtime type)"

import torch

print("Python:", sys.version.split()[0])
print("CUDA:", torch.cuda.is_available(), end="")
if torch.cuda.is_available():
    print(" —", torch.cuda.get_device_name(0))
else:
    print()

## 1. Project folder and clone (private repos)

**Private GitHub repo:** open the **key icon** (Secrets) in Colab, add a secret named **`GITHUB_TOKEN`**, and turn **Notebook access** on for it.

Create a [Personal Access Token](https://github.com/settings/tokens) (classic: **repo** scope, or fine-grained: **Contents → Read** on this repository).

Edit **`GIT_REPO_OWNER`** and **`GIT_REPO_NAME`** below to match your repo. The clone URL is built as `https://oauth2:<token>@github.com/owner/repo.git` (token never printed).

**Skip cloning:** if you upload/unzip the project yourself, set **`PROJECT_ROOT`** to that folder and set **`SKIP_CLONE = True`**.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
from urllib.parse import quote

# --- edit for your GitHub repo ---
GIT_REPO_OWNER = "maxraffel"
GIT_REPO_NAME = "attribute-based-watermarking"
SKIP_CLONE = False  # set True if you already put the repo under PROJECT_ROOT

PROJECT_ROOT = Path("/content") / GIT_REPO_NAME


def _github_token() -> str | None:
    try:
        from google.colab import userdata

        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return os.environ.get("GITHUB_TOKEN")


def _clone_url() -> str:
    token = _github_token()
    base = f"https://github.com/{GIT_REPO_OWNER}/{GIT_REPO_NAME}.git"
    if not token:
        print(
            "No GITHUB_TOKEN: using public HTTPS (private repos will fail). "
            "Add Colab secret GITHUB_TOKEN with repo read access."
        )
        return base
    return f"https://oauth2:{quote(token, safe='')}@github.com/{GIT_REPO_OWNER}/{GIT_REPO_NAME}.git"


def run_cmd(cmd: list[str] | str, *, cwd: Path | None = None) -> None:
    if isinstance(cmd, str):
        print("$", cmd)
        subprocess.run(cmd, shell=True, check=True, cwd=cwd)
    else:
        print("$", " ".join(cmd))
        subprocess.run(cmd, check=True, cwd=cwd)


if SKIP_CLONE:
    print("SKIP_CLONE=True — not cloning.")
elif not (PROJECT_ROOT / "watermarking.py").is_file():
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    run_cmd(["git", "clone", "--depth", "1", _clone_url(), str(PROJECT_ROOT)])
else:
    print("Already present:", PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("cwd:", os.getcwd())

## 2. Build CPRF (`cprf.so`)

Linux `.so` only — Colab builds it here (your Windows binary is not used).

In [ ]:
import shutil
import subprocess
from pathlib import Path

cprf_dir = PROJECT_ROOT / "cprf"
so_path = cprf_dir / "cprf.so"

if shutil.which("go") is None:
    subprocess.run(
        "apt-get update -qq && apt-get install -qq -y golang-go",
        shell=True,
        check=True,
    )

if not so_path.is_file():
    subprocess.run(
        ["go", "build", "-o", "cprf.so", "-buildmode=c-shared", "cprf.go"],
        cwd=cprf_dir,
        check=True,
    )

assert so_path.is_file(), "cprf.so missing"
print("CPRF:", so_path)

## 3. Build PRC (Rust + maturin)

Installs Rust in the VM home if `rustc` is missing, then `maturin develop` for the `prc` extension.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

cargo_bin = Path.home() / ".cargo" / "bin"
if not (cargo_bin / "rustc").exists():
    subprocess.run(
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y",
        shell=True,
        check=True,
    )
os.environ["PATH"] = str(cargo_bin) + os.pathsep + os.environ.get("PATH", "")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "maturin"], check=True)
subprocess.run(
    ["maturin", "develop", "--release", "-m", "prc/Cargo.toml"],
    cwd=PROJECT_ROOT,
    check=True,
)
print("PRC: maturin develop ok")

## 4. Python dependencies (Colab `%pip`)

Uses Colab’s recommended install path. Colab usually ships **PyTorch with CUDA**; extra packages match `pyproject.toml` (without the local-only CUDA index).

In [ ]:
%pip install -q "transformers==5.5.4" "rich>=15" "keybert>=0.9" sentencepiece accelerate

## 5. Hugging Face login (gated Llama)

Accept the **Llama 3.2** license on the model card, then run the cell below and paste a token with **read** access.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 6. Run the demo

Loads **Llama-3.2-1B-Instruct** and **DeBERTa-v3-large** NLI — first run downloads both; VRAM use can be high on a T4.

In [ ]:
import os
import runpy

os.chdir(PROJECT_ROOT)
runpy.run_path(str(PROJECT_ROOT / "app.py"), run_name="__main__")

## Optional: attribute test script

Uncomment in the next cell to run `test_attr_classification.py`.

In [ ]:
# import os, runpy
# os.chdir(PROJECT_ROOT)
# runpy.run_path(str(PROJECT_ROOT / "test_attr_classification.py"), run_name="__main__")